# Estimator Sheet 
This notebook builds the single combined dataset used for quantile regression estimation. It walks a folder of forecast-card CSVs (`project-*.csv`, `poi-*.csv`, `observations-*.csv`, and optional `forecast-*.csv` files), finds every project, combines each project's files into one merged table, adds K/D factors and peak-hour capacity/volume, and writes everything out to a single `combined_output.csv`. That combined file is the input for the model-fitting notebook.

**How to use this notebook:**
1. Run the "find project folders" cell to confirm every project was located. Set `DATA_ROOT` to the folder containing all your project CSVs.
2. Run the "build the combined dataset" cell. For each project this loads its project, poi, observations, and forecast CSVs, merges them together, and labels each observation as `before` or `after` the project's actual open date. All projects are then stacked into one `master_df`.
3. Run the "years from open" cell to add a `years_from_open` column to `master_df`, the difference between each observation's year and the project's actual open year.
4. Run the "K/D factors and peak hour capacity/volume" cell to add `k_factor`, `d_factor`, `peak_hour_capacity`, and `peak_hour_volume_observation`/`peak_hour_volume_forecast` to `master_df`, looked up from the factor tables (`K_D_Factors.csv`, `Cap_Base_Values.csv`, `Cap_Cal_Values.csv`) and matched on each row's `functional_class`.
5. Run the "spot-check" cell to visually confirm the merge and factor calculations worked as expected.
6. Run the "write out" cell to save `master_df` to `combined_output.csv`.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Find your project folders

Point `DATA_ROOT` at the folder holding your project data. This walks the entire directory tree underneath it and collects every sub-folder that contains a `project-*.csv` file (each matching folder is treated as one project). The result is a sorted list of folder paths, printed below so you can confirm the right projects were found.

**Update this path to point at your own data folder before continuing.**

In [ ]:
def find_project_folders(root_dir):
    """Return every sub-folder of root_dir that contains a project-*.csv file."""
    folders = []
    for dirpath, _dirnames, filenames in os.walk(root_dir):
        if any(re.match(r"project-.*\.csv$", f, re.IGNORECASE) for f in filenames):
            folders.append(dirpath)
    return sorted(folders)

DATA_ROOT = r"c:\Users\hmod225\Documents\Code\forecastcards_ba_data\data"
project_folders = find_project_folders(DATA_ROOT)
project_folders

## 2. Build the combined dataset

This section does the actual finding, loading, and merging for every project, then stacks the results into one master table used for estimation.

**Loading a project's files** — for a given project folder, finds and reads:
- `project-*.csv` → project-level metadata (one row per project)
- `poi-*.csv` → "points of interest" tied to that project (e.g. intersections/segments)
- `observations-*.csv` → observed traffic/volume records
- `forecast-*.csv` → **optional** — loaded if present, otherwise treated as missing

**Merging a project's files** — performs three sequential merges:
- **project → poi:** tags each POI row with its `project_id`, then merges project metadata onto every POI row (keeping all POIs, broadcasting the single project row across them).
- **before/after labeling:** converts `observation_date` to a real datetime and compares it to the project's `date_open_actual` to create `before_or_after`.
- **(project+poi) → observations:** merges the project/poi table onto the observations, matching on `poi_id` **and** `before_or_after`, keeping every observation row.
- **+ forecast:** if a forecast file exists, merges it in on `poi_id`, suffixing overlapping columns `_observation`/`_forecast`; otherwise renames `start_time`/`end_time` to the `_observation` versions so column naming stays consistent either way.

In [ ]:
def load_project_files(folder):
    """Load the project / poi / observations / (optional) forecast csvs for one folder."""
    project_path = glob.glob(os.path.join(folder, "project-*.csv"))[0]
    poi_path = glob.glob(os.path.join(folder, "poi-*.csv"))[0]
    obs_path = glob.glob(os.path.join(folder, "observations-*.csv"))[0]
    forecast_matches = glob.glob(os.path.join(folder, "forecast-*.csv"))

    project_df = pd.read_csv(project_path)
    poi_df = pd.read_csv(poi_path)
    obs_df = pd.read_csv(obs_path)
    forecast_df = pd.read_csv(forecast_matches[0]) if forecast_matches else None

    return project_df, poi_df, obs_df, forecast_df

In [ ]:
def process_project_folder(folder):
    project_df, poi_df, obs_df, forecast_df = load_project_files(folder)

    project_id = project_df["project_id"].iloc[0]
    date_open_actual = pd.to_datetime(project_df["date_open_actual"].iloc[0])

    poi_df = poi_df.copy()

    if "before_or_after" in poi_df.columns:
        poi_df["before_or_after"] = (
            poi_df["before_or_after"].astype("string").str.strip().str.lower()
        )
    poi_df["project_id"] = project_id

    # STEP 1: project -> poi
    project_poi = pd.merge(project_df, poi_df, on="project_id", how="right")

    # derive a sensible before/after label for each observation
    obs_df = obs_df.copy()
    obs_df["observation_date"] = pd.to_datetime(obs_df["observation_date"], errors="coerce")
    obs_df["before_or_after"] = pd.array(
        np.where(obs_df["observation_date"] < date_open_actual, "before", "after"),
        dtype="string",
    )
    obs_df["before_or_after_binary"] = (obs_df["before_or_after"] == "after").astype(int)

    # STEP 2: (project + poi) -> observation
    merged = pd.merge(
        project_poi, obs_df, on=["poi_id", "before_or_after"], how="right"
    )

    # STEP 3: + forecast
    if forecast_df is not None:
        merged = pd.merge(
            merged, forecast_df, on="poi_id", how="left",
            suffixes=("_observation", "_forecast")
        )
    else:
        merged = merged.rename(columns={"start_time": "start_time_observation",
                                         "end_time": "end_time_observation"})

    return merged

In [ ]:
project_tables = [process_project_folder(folder) for folder in project_folders]

master_df = pd.concat(project_tables, ignore_index=True, sort=False)

print(f"{len(master_df)} total observation rows across {len(project_tables)} project(s)")
master_df.head(10)

## 3. Years from open (before/after completion year)

Adds a `years_from_open` column to `master_df`: the difference between the calendar year of each observation and the calendar year the project actually opened (`date_open_actual`).

In [ ]:
master_df["observation_date"] = pd.to_datetime(master_df["observation_date"], errors="coerce")
master_df["date_open_actual"] = pd.to_datetime(master_df["date_open_actual"], errors="coerce")

master_df["years_from_open"] = (
    master_df["observation_date"].dt.year - master_df["date_open_actual"].dt.year
)

master_df[["project_id", "poi_id", "observation_date", "date_open_actual", "years_from_open"]].head(10)

## 4. K/D factors and peak hour capacity/volume

Adds four columns using the factor tables (`K_D_Factors.csv`, `Cap_Base_Values.csv`, `Cap_Cal_Values.csv`), matched to each row's `functional_class`:

- `k_factor`, `d_factor` — looked up directly from `K_D_Factors.csv`.
- `peak_hour_capacity` — HCM-style calc: `Capacity = Cap_base x N x f_hv x PHF x f_p x f_g`, where `Cap_base` comes from `Cap_Base_Values.csv`, `N` is `through_lanes`, and `f_hv`/`PHF`/`f_p`/`f_g` come from `Cap_Cal_Values.csv`.
- `peak_hour_volume_observation` / `peak_hour_volume_forecast` — `AADT x K x D`, computed separately for the observed and forecast volume columns (only where the unit is `AADT`).

Rows with `functional_class == "unknown"` (or missing `through_lanes`) will come out blank for these columns.

In [ ]:
# Paths to the factor tables
FACTOR_TABLES_ROOT = r"c:\Users\hmod225\Documents\Code\forecastcards_v2\forecastcards\capacity_factor_tables"
K_D_FACTORS_PATH = os.path.join(FACTOR_TABLES_ROOT, "K_D_Factors.csv")
CAP_BASE_PATH = os.path.join(FACTOR_TABLES_ROOT, "Cap_Base_Values.csv")
CAP_CAL_PATH = os.path.join(FACTOR_TABLES_ROOT, "Cap_Cal_Values.csv")

def normalize_functional_class(value):
    """Lowercase/trim a functional class label so it matches across files
    (e.g. K_D_Factors.csv uses 'Local', Cap_Base_Values.csv uses 'Local Road')."""
    if pd.isna(value):
        return None
    label = str(value).strip().lower()
    if label == "local road":
        label = "local"
    return label

# K / D factors by functional class
kd_factors = pd.read_csv(K_D_FACTORS_PATH)[["Functional Class", "K Factor", "D Factor"]].dropna(subset=["Functional Class"])
kd_factors["_fc"] = kd_factors["Functional Class"].apply(normalize_functional_class)
k_factor_lookup = kd_factors.set_index("_fc")["K Factor"].to_dict()
d_factor_lookup = kd_factors.set_index("_fc")["D Factor"].to_dict()

# base capacity (veh/hr/lane) by functional class
cap_base = pd.read_csv(CAP_BASE_PATH, thousands=",")
cap_base["_fc"] = cap_base["Facility Type"].apply(normalize_functional_class)
cap_base_lookup = cap_base.set_index("_fc")["Base Capacity (veh/hr/lane)"].to_dict()

# scalar HCM adjustment factors (f_hv, PHF, f_p, f_g)
cap_params = pd.read_csv(CAP_CAL_PATH).set_index("Symbol")["Default Value"].to_dict()
f_hv, PHF, f_p, f_g = cap_params["f_hv"], cap_params["PHF"], cap_params["f_p"], cap_params["f_g"]

k_factor_lookup, d_factor_lookup, cap_base_lookup, cap_params

In [ ]:
# make sure through_lanes is numeric
master_df["through_lanes"] = pd.to_numeric(master_df["through_lanes"], errors="coerce")

_fc_norm = master_df["functional_class"].apply(normalize_functional_class)
master_df["k_factor"] = _fc_norm.map(k_factor_lookup)
master_df["d_factor"] = _fc_norm.map(d_factor_lookup)
_cap_base = _fc_norm.map(cap_base_lookup)

# Capacity = Cap_base x N x f_hv x PHF x f_p x f_g
master_df["peak_hour_capacity"] = _cap_base * master_df["through_lanes"] * f_hv * PHF * f_p * f_g

# Peak hour volume = AADT x K x D, for both the observed and forecast volume columns
master_df["peak_hour_volume_observation"] = np.where(
    master_df["observation_unit"] == "AADT",
    master_df["observation_value"] * master_df["k_factor"] * master_df["d_factor"],
    np.nan,
)
master_df["peak_hour_volume_forecast"] = np.where(
    master_df["forecast_unit"] == "AADT",
    master_df["forecast_value"] * master_df["k_factor"] * master_df["d_factor"],
    np.nan,
)

master_df[[
    "functional_class", "through_lanes", "k_factor", "d_factor", "peak_hour_capacity",
    "observation_value", "peak_hour_volume_observation",
    "forecast_value", "peak_hour_volume_forecast",
]].head(10)

## 5. Spot-check the merged data

Displays a narrowed view of just the columns most relevant for spot-checking, the join keys, dates, before/after labels, and a few geometric attributes (lane counts, shoulder width), to visually verify the merges worked correctly before writing anything out.

In [ ]:
master_df[[
    "project_id", "poi_id", "observation_date", "date_open_actual",
    "before_or_after", "before_or_after_binary", "years_from_open",
    "through_lanes", "right_turn_lanes", "shoulder_width",
    "functional_class", "k_factor", "d_factor", "peak_hour_capacity",
    "peak_hour_volume_observation", "peak_hour_volume_forecast",
]]

## 6. Write out the combined CSV

Saves `master_df` to `combined_output.csv` inside `DATA_ROOT` (the same folder your project subfolders live in, alongside `validation_report.txt`) and confirms how many rows/columns were written. This file is the input for the quantile regression model-fitting notebook.

In [ ]:
output_path = os.path.join(DATA_ROOT, "combined_output.csv")
master_df.to_csv(output_path, index=False)
print(f"Wrote {len(master_df)} rows and {len(master_df.columns)} columns to {output_path}")